In [1]:
!pip install LiteEFG numpy pandas tqdm

In [2]:
import LiteEFG as efg
from LiteEFG.baselines.CFR import graph as CFR
import pandas as pd
from tqdm import trange
# Load game and register CFR
game_path = "sam_sead.game"
env = efg.FileEnv(game_path, traverse_type="Enumerate")
alg = CFR()
env.set_graph(alg)
print("SAM-SEAD game loaded. CFR graph registered.")

===============Graph is ready for CFR===============


SAM-SEAD game loaded. CFR graph registered.


# SAM-SEAD Imperfect Information Extensive Form Game

## Scenario
A **Blue SEAD aircraft** is approaching a **Red SAM battery**.
This is a 2-level sequential game with imperfect information:

```
Level 1 — SAM (Red) decides: Radar ON or Radar OFF
           (This is hidden from SEAD — SEAD cannot see the radar state)

Level 2 — SEAD (Blue) decides: Attack or Hold
           (SEAD can only observe signals, not the true radar state)
```

## Imperfect Information
SEAD cannot directly observe whether the SAM radar is ON or OFF.
Both Level-2 SEAD decision nodes (facing ON or facing OFF)
are grouped into a **single infoset** — SEAD must act without
knowing the true radar state.

## Payoffs (Blue perspective; Red = −Blue, zero-sum)

| SAM state | SEAD action | Outcome | Blue | Red |
|-----------|-------------|---------|------|-----|
| Radar ON  | Attack      | SEAD fires ARM → destroys SAM; but SAM may score hit | +4 | −4 |
| Radar ON  | Hold        | SAM radar active, tracks & engages SEAD aircraft | −5 | +5 |
| Radar OFF | Attack      | SEAD fires ARM blindly → misses; wasted ordnance | −2 | +2 |
| Radar OFF | Hold        | SAM stays silent; neither side gains | 0  | 0  |

### Payoff reasoning
- **ON + Attack (+4):** SEAD detects emission and fires ARM. High Blue gain — SAM destroyed.
- **ON + Hold (−5):** SAM is active and SEAD does nothing — SAM engages Blue aircraft. Worst Blue outcome.
- **OFF + Attack (−2):** SEAD fires blind (no emission to home on) — wastes missile, exposes position.
- **OFF + Hold (0):** SAM stays dark, SEAD holds fire — a tense standoff with no exchange.

In [2]:
# Write the SAM-SEAD IIEFG game file
# Player 1 = SAM (Red)  — moves first, chooses radar ON or OFF
# Player 2 = SEAD (Blue) — moves second, cannot observe SAM's choice

game_content = """# SAM-SEAD Imperfect Information EFG
#
# Opt {
#     game_tree: true,
#     num_players: 2,
#     depth: 2,
# }
#
node / player 1 actions on off
node /P1:on  player 2 actions attack hold
node /P1:off player 2 actions attack hold
node /P1:on/P2:attack  leaf payoffs 1=-4 2=4
node /P1:on/P2:hold    leaf payoffs 1=5  2=-5
node /P1:off/P2:attack leaf payoffs 1=2  2=-2
node /P1:off/P2:hold   leaf payoffs 1=0  2=0
infoset pl2_0__ nodes /P1:on /P1:off
"""

game_path = "sam_sead.game"
with open(game_path, "w") as f:
    f.write(game_content)
print(f"Written to: {game_path}")

Written to: sam_sead.game


In [ ]:
import LiteEFG as efg
game_path = "sam_sead.game"
env = efg.FileEnv(game_path, traverse_type="Enumerate")
print("FileEnv OK")

In [3]:
from LiteEFG.baselines.CFR import graph as CFR
alg = CFR()
print("CFR OK")

===============Graph is ready for CFR===============


CFR OK


In [ ]:
# Run CFR
NUM_ITER   = 10000
PRINT_FREQ = 1000
history    = []
best_exp   = 1e9

for i in trange(NUM_ITER, desc="CFR (SAM-SEAD)"):
    alg.update_graph(env)
    env.update_strategy(alg.current_strategy(), update_best=False)

    if i % PRINT_FREQ == 0 or i == NUM_ITER - 1:
        explo = sum(env.exploitability(alg.current_strategy(), "avg-iterate"))
        best_exp = min(best_exp, explo)
        history.append({"iteration": i, "exploitability": round(explo, 8), "best": round(best_exp, 8)})

print("\nDone.")

In [ ]:
# Convergence
print("--- Exploitability convergence ---")
print(pd.DataFrame(history).to_string(index=False))

In [ ]:
# Nash Equilibrium mixed strategies
print("=" * 60)
labels = {
    1: {"name": "SAM  (Red)",  "actions": ["Radar ON", "Radar OFF"]},
    2: {"name": "SEAD (Blue)", "actions": ["Attack",   "Hold"]},
}

for player_id in range(1, 3):
    info = labels[player_id]
    print(f"\nPlayer {player_id} — {info['name']}  |  Nash Equilibrium (avg-iterate):")
    strategy_pairs = env.get_strategy(player_id, alg.current_strategy(), "avg-iterate")
    for infoset_name, probs in strategy_pairs:
        print(f"  Infoset: {infoset_name}")
        for action, p in zip(info['actions'], probs):
            print(f"    {action:12s}: {p:.4f}  ({p*100:.1f}%)")
print("=" * 60)

print("""
Interpretation:
  SAM  (Red)  must mix ON/OFF to prevent SEAD from exploiting a pure strategy.
  SEAD (Blue) must mix Attack/Hold for the same reason.

  If SAM always turns ON  -> SEAD always Attacks  -> SAM loses badly.
  If SAM always turns OFF -> SEAD always Holds    -> SAM gets a free pass but loses threat value.
  The Nash mix makes both players indifferent between their actions.

  Expected game value (Blue) at Nash:
    Let p = P(SAM ON), q = P(SEAD Attack)
    V = p*q*(+4) + p*(1-q)*(-5) + (1-p)*q*(-2) + (1-p)*(1-q)*(0)
""")

# Compute and print expected game value at Nash
s_on  = strategy_pairs = env.get_strategy(1, alg.current_strategy(), "avg-iterate")
s_on  = dict(env.get_strategy(1, alg.current_strategy(), "avg-iterate"))
s_sea = dict(env.get_strategy(2, alg.current_strategy(), "avg-iterate"))

p_on  = list(s_on.values())[0][0]   # P(SAM ON)
q_att = list(s_sea.values())[0][0]  # P(SEAD Attack)

V_blue = (p_on * q_att * 4
        + p_on * (1 - q_att) * (-5)
        + (1 - p_on) * q_att * (-2)
        + (1 - p_on) * (1 - q_att) * 0)

print(f"  p(SAM ON)      = {p_on:.4f}")
print(f"  p(SEAD Attack) = {q_att:.4f}")
print(f"  Expected value for Blue (SEAD) at Nash = {V_blue:.4f}")
print(f"  Expected value for Red  (SAM)  at Nash = {-V_blue:.4f}")